# Regression Analysis

I fit OLS models with WTI returns at lag 3 and VIX levels at lag 2. Because those lags were chosen using the same dataset, the model is exploratory rather than an independent prediction test.

In [ ]:
from src.regression import (
    build_regression_dataset,
    run_ols,
)

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

In [ ]:
regression = build_regression_dataset()

regression.head()

## Regression Dataset

Built from the lagged predictors selected in the lead-lag analysis: ship return as the dependent variable, with WTI return (lag 3) and VIX level (lag 2) as the independent variables.

In [ ]:
model_wti = run_ols(
    regression,
    ["wti_return_lag3"]
)

model_vix = run_ols(
    regression,
    ["vix_level_lag2"]
)

model_combined = run_ols(
    regression,
    [
        "wti_return_lag3",
        "vix_level_lag2"
    ]
)

In [ ]:
summary = pd.DataFrame(
    [
        {
            "Model": "WTI Lag 3",
            "Predictor": "WTI Return Lag 3",
            "Coefficient": model_wti.params["wti_return_lag3"],
            "P-value": model_wti.pvalues["wti_return_lag3"],
            "R-squared": model_wti.rsquared,
            "Adj. R-squared": model_wti.rsquared_adj,
        },
        {
            "Model": "VIX Lag 2",
            "Predictor": "VIX Level Lag 2",
            "Coefficient": model_vix.params["vix_level_lag2"],
            "P-value": model_vix.pvalues["vix_level_lag2"],
            "R-squared": model_vix.rsquared,
            "Adj. R-squared": model_vix.rsquared_adj,
        },
        {
            "Model": "Combined",
            "Predictor": "WTI Return Lag 3",
            "Coefficient": model_combined.params["wti_return_lag3"],
            "P-value": model_combined.pvalues["wti_return_lag3"],
            "R-squared": model_combined.rsquared,
            "Adj. R-squared": model_combined.rsquared_adj,
        },
        {
            "Model": "Combined",
            "Predictor": "VIX Level Lag 2",
            "Coefficient": model_combined.params["vix_level_lag2"],
            "P-value": model_combined.pvalues["vix_level_lag2"],
            "R-squared": model_combined.rsquared,
            "Adj. R-squared": model_combined.rsquared_adj,
        },
    ]
)

summary.round(3)

## Regression Results

WTI lag 3 has an unadjusted p-value of 0.042 in the combined regression. This does not adjust for the earlier lag search, and explanatory power is limited (R² = 3.8%).

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    model_combined.fittedvalues,
    model_combined.resid,
    alpha=0.7
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

Residuals scatter randomly around zero with no obvious nonlinear pattern, which supports using a linear model here.

In [ ]:
sm.qqplot(
    model_combined.resid,
    line="s"
)

plt.title("Normal Q-Q Plot")

plt.show()

Residuals deviate slightly from normality — not unusual for financial return data, which tends to have heavier tails than a normal distribution would suggest.

In [ ]:
bp = het_breuschpagan(
    model_combined.resid,
    model_combined.model.exog
)

bp_result = pd.DataFrame({

    "Statistic":[
        bp[0],
        bp[2]
    ],

    "P-value":[
        bp[1],
        bp[3]
    ]

},
index=[
    "LM Test",
    "F Test"
])

bp_result.round(3)

No sign of heteroskedasticity, so the constant-variance assumption behind OLS holds up fine here.

In [ ]:
X = regression[
    [
        "wti_return_lag3",
        "vix_level_lag2"
    ]
]

X = sm.add_constant(X)

vif = pd.DataFrame()

vif["Variable"] = X.columns

vif["VIF"] = [

    variance_inflation_factor(
        X.values,
        i
    )

    for i in range(X.shape[1])

]

vif.round(3)

Both predictors have VIF values close to one, so multicollinearity isn't a concern — each variable is contributing distinct information.

In [ ]:
dw = durbin_watson(
    model_combined.resid
)

print(
    f"Durbin-Watson Statistic: {dw:.3f}"
)

Durbin–Watson is around 1.40, suggesting positive autocorrelation in residuals. This can affect standard OLS inference; robust standard errors or a time-series model would be worth checking.

## Key Findings

The combined regression yields WTI lag 3 β = 0.059 (raw p = 0.042) and VIX lag 2 β ≈ -0.0008 (raw p = 0.108). The nominal F-test p-value is 0.027, with R² = 3.8% and adjusted R² = 2.7%. Because lag selection and model fitting used the same sample, and residuals show autocorrelation, these p-values should be treated as exploratory.

## Business Conclusion

The WTI association is worth studying further, but it is not yet a validated early-warning signal. A broader predictor set, time-series-aware inference, and out-of-sample evaluation would help assess whether the relationship persists.